In [1]:
import os
# For me, without this, it takes a long time to initialize
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"
import csv
import copy
import cv2
import mediapipe as mp
import numpy as np
import itertools

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

HF_TOKEN = os.getenv('HF_TOKEN')

In [35]:
def logging_csv(number, landmark_list):
    csv_path = 'model/keypoint_classifier/keypoint.csv'
    with open(csv_path, 'a', newline="") as f:
        writer = csv.writer(f)
        writer.writerow([number, *landmark_list])
    return

def calc_bounding_rect(image, landmarks):
    # print(type(landmarks))
    # print(dir(landmarks))
    # print(landmarks)
    image_width, image_height = image.shape[1], image.shape[0]

    landmark_array = np.empty((0, 2), int)

    for _, landmark in enumerate(landmarks):
        landmark_x = min(int(landmark.x * image_width), image_width - 1)
        landmark_y = min(int(landmark.y * image_height), image_height - 1)

        landmark_point = [np.array((landmark_x, landmark_y))]

        landmark_array = np.append(landmark_array, landmark_point, axis=0)

    x, y, w, h = cv2.boundingRect(landmark_array)

    return [x, y, x + w, y + h]


def calc_landmark_list(image, landmarks):
    image_width, image_height = image.shape[1], image.shape[0]

    landmark_point = []

    # Keypoint
    for _, landmark in enumerate(landmarks):
        landmark_x = min(int(landmark.x * image_width), image_width - 1)
        landmark_y = min(int(landmark.y * image_height), image_height - 1)
        # landmark_z = landmark.z

        landmark_point.append([landmark_x, landmark_y])

    return landmark_point


def pre_process_landmark(landmark_list):
    temp_landmark_list = copy.deepcopy(landmark_list)

    # Convert to relative coordinates
    base_x, base_y = 0, 0
    for index, landmark_point in enumerate(temp_landmark_list):
        if index == 0:
            base_x, base_y = landmark_point[0], landmark_point[1]

        temp_landmark_list[index][0] = temp_landmark_list[index][0] - base_x
        temp_landmark_list[index][1] = temp_landmark_list[index][1] - base_y

    # Convert to a one-dimensional list
    temp_landmark_list = list(
        itertools.chain.from_iterable(temp_landmark_list))

    # Normalization
    max_value = max(list(map(abs, temp_landmark_list)))

    def normalize_(n):
        return n / max_value

    temp_landmark_list = list(map(normalize_, temp_landmark_list))

    return temp_landmark_list


In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("Samarth0710/bharatanatyam-mudra-dataset")

# Access the data
train_data = dataset["train"]
print(f"Number of samples: {len(train_data)}")
print(f"Features: {train_data.features}")

# Example: Get first image and label
sample = train_data[0]
image = sample["image"]
label = sample["label"]
print(f"Label: {label}")

c:\Users\theke\Documents\Code\bharatanatyam-mudra-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of samples: 28431
Features: {'image': Image(mode=None, decode=True), 'label': Value('string'), 'gesture_type': Value('string'), 'label_id': Value('int64')}
Label: Sikharam


In [ ]:
# Focusing on single hand gestures for now
single_hand_train = train_data.filter(lambda x: x["gesture_type"] == 'single_hand')
print(f"Number of samples: {len(single_hand_train)}")

Filter: 100%|██████████| 28431/28431 [00:22<00:00, 1260.63 examples/s]

Number of samples: 14827


In [ ]:
single_hand_unique_labels = single_hand_train.unique('label')
single_hand_label_id_map = {}
for i, label in enumerate(single_hand_unique_labels):
    single_hand_label_id_map[label] = i

Flattening the indices: 100%|██████████| 14827/14827 [00:00<00:00, 26784.99 examples/s]


['Sikharam',
 'Tamarachudam',
 'Sarpasirsha',
 'Katakamukha_1',
 'Kangulam',
 'Tripathaka',
 'Mukulam',
 'Chandrakala',
 'Suchi',
 'Chaturam',
 'Pathaka',
 'Mayura',
 'Kapith',
 'Mrigasirsha',
 'Hamsasyam',
 'Mushti',
 'Ardhachandran',
 'Shukatundam',
 'Katakamukha_2',
 'Aralam',
 'Katrimukha',
 'Simhamukham',
 'Katakamukha_3',
 'Ardhapathaka',
 'Trishulam',
 'Bramaram',
 'Padmakosha',
 'Alapadmam']

In [ ]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Initialize landmarker
base_options = python.BaseOptions(model_asset_path='googletutorial/hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
)
detector = vision.HandLandmarker.create_from_options(options)

In [47]:
split = single_hand_train.train_test_split(test_size=0.2)
train_dataset = split["train"]
test_dataset = split["test"]
# sample_train = train_dataset.select(range(100))
sample_train = single_hand_train.select(range(100))

In [ ]:
valid_count = 0
for i, row in enumerate(train_dataset):
    row_image = row['image']
    row_label = row['label']
    number = single_hand_label_id_map[row_label]
    # print(row_label)

    debug_image = copy.deepcopy(row_image)
    img_array = np.array(row_image)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_array)
    rgb_frame = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
    mp_frame = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    detection_result = detector.detect(mp_frame)
    #  ####################################################################
    # print(dir(detection_result))
    # print(detection_result.hand_landmarks)
    # print(detection_result.handedness)
    annotated_image = mp_image.numpy_view()   
    if detection_result.hand_landmarks is not None:
        # if not detection_result.hand_landmarks:
            # print('No landmarks found!')
            # print('index:', i)
            # row_image.show()
            # break
        for hand_landmarks, handedness in zip(detection_result.hand_landmarks,
                                                detection_result.handedness):
            # print('Landmarks found')
            # # Bounding box calculation
            # brect = calc_bounding_rect(img_array, hand_landmarks)
            # Landmark calculation
            landmark_list = calc_landmark_list(img_array, hand_landmarks)

            # Conversion to relative coordinates / normalized coordinates
            pre_processed_landmark_list = pre_process_landmark(
                landmark_list)
            # print('calling logging')
            logging_csv(number, pre_processed_landmark_list)
            valid_count += 1
valid_ratio = valid_count/len(train_dataset)
print(f'valid ratio: {valid_ratio * 100}%')

valid ratio: 24.25


In [53]:
len(train_dataset)

11861

One problem I'm seeing is that like 70-80% of the pictures don't get recognized by the landmark detector. Maybe that's not a problem since there's so much data though

In [57]:
for label in single_hand_unique_labels:
    csv_path = 'model/keypoint_classifier/keypoint_classifier_label.csv'
    with open(csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([label])